# Meet the Synthetic-Data Sandbox

**Week 1 · Lesson 9** &nbsp;·&nbsp; GBUS 564: Predictive Analytics &nbsp;·&nbsp; Fall 2026

<a target="_blank" href="https://colab.research.google.com/github/avorybryant/umw_gbus564_fall26/blob/main/notebooks/week1/lectures/09_synthetic_data_sandbox.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

## Setup

Same boilerplate as every notebook this term — see Lesson 8 if this is your
first time seeing it.

In [ ]:
# Environment setup — identical across every notebook in the course.
# Installs the course package (dataset loaders + helpers) from the public
# course repo; Colab already has numpy/pandas/scikit-learn, so it's quick.
%pip install -q matplotlib git+https://github.com/avorybryant/umw_gbus564_fall26.git

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn import set_config

set_config(display="diagram")
plt.rcParams["figure.figsize"] = (8, 5)

RANDOM_SEED = 564  # GBUS 564 — reuse this seed everywhere for reproducibility
rng = np.random.default_rng(RANDOM_SEED)

## 1. Business Problem

**Why this matters:** Weeks 2, 3, 4, 6, and 7 each deliberately break a model
assumption — an overfit polynomial, coefficients gone unstable under
multicollinearity, k-means's spherical-cluster assumption, an
over-parameterized neural net, a time series validated with a leaky random
split. Real data won't reliably hand you a clean, isolated version of any of
these on demand — it's noisy, but not noisy *exactly the amount* you need to
make the point, and you can't dial its correlation structure up or down. A
parameterized generator can.

This notebook is the tour: four functions in `gbus564_coursework.utils.sandbox`, one data
shape each, dialed with **noise**, **imbalance**, **confounding**, and
**drift**. Later lessons don't re-explain these functions — they just call
them with different knob settings.

## 2. Data

Each generator takes the same shared `rng` as everything else in this
course. They live in `gbus564_coursework.utils.sandbox`, imported directly
in the first code cell below — there's no separate loading step, since
these are generated, not loaded. The rest of this section is a quick tour
of each generator — what knob it exposes, and which later lesson dials it.

### Regression sandbox — noise

`make_regression_sandbox` returns a true nonlinear signal (a cubic) plus
Gaussian noise on one feature:

$$y = 0.5x^3 - 2x + \text{noise} \cdot \varepsilon, \quad \varepsilon \sim \mathcal{N}(0, 1)$$

Turning up `noise` is what lets **Week 2
Lesson 9 (Overfitting in the Sandbox)** show a high-degree polynomial chase
individual noisy points instead of the underlying curve.

In [ ]:
from gbus564_coursework.utils.sandbox import (
    make_regression_sandbox,
    make_classification_sandbox,
    make_clusters_sandbox,
    make_timeseries_sandbox,
)

fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
for ax, noise in zip(axes, [0.5, 3.0]):
    X, y = make_regression_sandbox(n_samples=150, noise=noise, rng=rng)
    ax.scatter(X["x"], y, s=10, alpha=0.6)
    ax.set_title(f"noise={noise}")
    ax.set_xlabel("x")
axes[0].set_ylabel("y")
plt.tight_layout()

### Regression sandbox — confounding

The same function, dialed differently: `n_features > 1` with
`confound=True` draws pairwise-correlated features (ρ≈0.95) instead of one.
**Week 3 Lesson 6 (The Multicollinearity Problem)** refits this on bootstrap
resamples and watches coefficients swing wildly — even flip sign — because
the model can't tell correlated features apart.

Two separate equations are at play, and it's easy to conflate them:

$$X \sim \mathcal{N}(\mathbf{0}, \Sigma), \qquad \Sigma_{ij} = \begin{cases} 1 & i = j \\ \rho & i \ne j \end{cases}$$

$$y = \beta_0 x_0 + \beta_1 x_1 + \beta_2 x_2 + 0 \cdot x_3 + 0 \cdot x_4 + \text{noise} \cdot \varepsilon, \quad \varepsilon \sim \mathcal{N}(0, 1)$$

`Σ` (feature-to-feature) is what `confound` toggles — `ρ=0` draws the 5
columns independently, `ρ=0.95` draws them as near-duplicates of each
other. `β` (feature-to-outcome, drawn once from `Uniform(1, 3)` on the
first `n_informative` entries) stays exactly the same either way — the true
relationship between `X` and `y` never changes. Only how correlated the
`X` columns are with *each other* does. That's what isolates multicollinearity
as the cause of the instability below: nothing about the signal itself
changed, only the geometry of the design matrix.</cell id="acf1a095">


In [ ]:
from sklearn.linear_model import LinearRegression


def bootstrap_coefs(X, y, n_boot=20):
    coefs = []
    for _ in range(n_boot):
        idx = rng.integers(0, len(X), size=len(X))
        fit = LinearRegression().fit(X.iloc[idx], y.iloc[idx])
        coefs.append(fit.coef_)
    return np.array(coefs)


X_indep, y_indep = make_regression_sandbox(n_samples=60, n_features=5, confound=False, rng=rng)
X_corr, y_corr = make_regression_sandbox(n_samples=60, n_features=5, confound=True, rng=rng)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5), sharex=True, sharey=True)
axes[0].scatter(X_indep["x0"], X_indep["x1"], s=10, alpha=0.6)
axes[0].set_title("confound=False — x0 vs x1")
axes[0].set_xlabel("x0")
axes[0].set_ylabel("x1")
axes[1].scatter(X_corr["x0"], X_corr["x1"], s=10, alpha=0.6)
axes[1].set_title("confound=True — x0 vs x1")
axes[1].set_xlabel("x0")
plt.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
axes[0].boxplot(bootstrap_coefs(X_indep, y_indep), tick_labels=X_indep.columns)
axes[0].set_title("confound=False — stable coefficients")
axes[1].boxplot(bootstrap_coefs(X_corr, y_corr), tick_labels=X_corr.columns)
axes[1].set_title("confound=True — unstable, sign-flipping")
plt.tight_layout()

### Classification sandbox — noise & imbalance

`make_classification_sandbox` returns a single score feature
(`n_features=1`) with class separation controlled by `noise` and class
balance controlled by `imbalance`. **Week 2 Lesson 5 (The Threshold/Stump)**
slides a cutoff across this score to trace out ROC/PR curves; **Week 2
Lesson 7 (Baselines)** uses the `imbalance` knob to show the accuracy
paradox — a majority-class baseline already scores 95% at 5% imbalance
without learning anything.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
for ax, noise in zip(axes, [0.5, 2.5]):
    X, y = make_classification_sandbox(n_samples=400, noise=noise, imbalance=0.3, rng=rng)
    ax.hist(X.loc[y == 0, "score"], bins=20, alpha=0.6, label="class 0")
    ax.hist(X.loc[y == 1, "score"], bins=20, alpha=0.6, label="class 1")
    ax.set_title(f"noise={noise}")
axes[0].legend()
plt.tight_layout()

X, y = make_classification_sandbox(n_samples=1000, imbalance=0.05, rng=rng)
print(f"imbalance=0.05 -> actual positive rate: {y.mean():.1%}")

In [ ]:
# Same score distributions, but `imbalance` sets how many of each class get
# drawn — the positive class shrinks while its location and spread stay put.
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharex=True, sharey=True)
for ax, imbalance in zip(axes, [0.5, 0.1]):
    X, y = make_classification_sandbox(n_samples=400, noise=0.3, imbalance=imbalance, rng=rng)
    ax.hist(X.loc[y == 0, "score"], bins=20, alpha=0.6, label="class 0")
    ax.hist(X.loc[y == 1, "score"], bins=20, alpha=0.6, label="class 1")
    ax.set_title(f"imbalance={imbalance}  (n_pos={int((y == 1).sum())})")
axes[0].legend()
plt.tight_layout()

### Clusters sandbox — shape

`make_clusters_sandbox` switches between three layouts: `"blobs"`
(spherical, k-means-friendly), `"moons"`, and `"anisotropic"` (both violate
the spherical-cluster assumption k-means relies on). **Week 4 Lesson 12**
fits k-means on all three — near-perfect on blobs, badly wrong on moons — to
make the assumption itself visible.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, shape in zip(axes, ["blobs", "moons", "anisotropic"]):
    X, labels = make_clusters_sandbox(n_samples=300, shape=shape, rng=rng)
    ax.scatter(X["x0"], X["x1"], c=labels, s=10, cmap="viridis")
    ax.set_title(shape)
plt.tight_layout()

### Time-series sandbox — drift

`make_timeseries_sandbox` builds a daily series with weekly seasonality
plus an optional level shift (`drift`) about 70% of the way through — a
regime change, the kind a monitor should catch (the W1 Lesson 5 "deploy →
monitor" thread). With `drift=0`, the same autocorrelated shape is what
**Week 7 Lesson 3** uses to show that shuffling a time series before
splitting inflates validation metrics — Sections 3–5 below run that
demonstration directly.

In [ ]:
y = make_timeseries_sandbox(n_periods=365, trend=0.05, seasonality=3.0, drift=8.0, rng=rng)

fig, ax = plt.subplots()
ax.plot(y.index, y.values)
ax.axvline(y.index[int(len(y) * 0.7)], color="red", linestyle="--", label="drift point")
ax.set_title("Daily series with weekly seasonality + a level-shift at 70%")
ax.legend()
plt.tight_layout()

### Time-series sandbox — reading each argument

The plot above stacks every knob at once. It's easier to read the arguments
one at a time: start from a flat line (`10`, the constant baseline) and turn
on one argument per panel.

- **`trend`** — linear slope per day. `0.05` climbs ~18 over the 365-day span.
- **`seasonality`** — the *amplitude* of the weekly sine, not its period. The
  cycle is always 7 days (one full up-and-down); `seasonality=3` swings ±3
  around the trend line.
- **`noise`** — std-dev of the daily Gaussian jitter layered on top.
- **`drift`** — a one-time level shift applied from 70% of the way through
  onward (red dotted line). `drift=8` steps the whole series up 8 and holds
  it there; trend and seasonality carry on unchanged.

All the terms are simply **added together**:
$y = 10 + \text{trend}\cdot t + \text{seasonal} + \text{drift}\cdot[t \ge 0.7n] + \text{noise}\cdot\varepsilon$.

In [ ]:
# Same series, built up one argument at a time — each panel adds exactly one
# knob on top of the panel above it. The noise draw is held fixed (a fresh
# rng seeded identically per panel) so only the toggled argument moves.
steps = [
    ("baseline — all knobs at 0", dict(trend=0.0, seasonality=0.0, noise=0.0, drift=0.0)),
    ("+ trend = 0.05", dict(trend=0.05, seasonality=0.0, noise=0.0, drift=0.0)),
    ("+ seasonality = 3", dict(trend=0.05, seasonality=3.0, noise=0.0, drift=0.0)),
    ("+ noise = 1", dict(trend=0.05, seasonality=3.0, noise=1.0, drift=0.0)),
    ("+ drift = 8", dict(trend=0.05, seasonality=3.0, noise=1.0, drift=8.0)),
]

fig, axes = plt.subplots(len(steps), 1, figsize=(9, 11), sharex=True, sharey=True)
for ax, (label, knobs) in zip(axes, steps):
    s = make_timeseries_sandbox(n_periods=365, rng=np.random.default_rng(RANDOM_SEED), **knobs)
    ax.plot(s.index, s.values, lw=0.9)
    ax.axvline(s.index[int(len(s) * 0.7)], color="red", linestyle=":", lw=1, alpha=0.6)
    ax.set_title(label, loc="left", fontsize=10)
axes[-1].set_xlabel("date")
plt.tight_layout()

## 3. Model

Section 3 usually opens with a baseline before the model under study; here
the point isn't the model, it's the **splitting strategy**, so the same
model gets fit twice — once on a random train/test split, once on a
temporal split — and Section 4 compares them. This time the series has
`drift=0`: an isolated look at autocorrelation leakage, without a regime
change mixed in.

In [ ]:
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split

y = make_timeseries_sandbox(n_periods=365, trend=0.05, seasonality=3.0, drift=0.0, rng=rng)
lags = pd.DataFrame({"y": y, "lag1": y.shift(1), "lag7": y.shift(7)}).dropna()
X, target = lags[["lag1", "lag7"]], lags["y"]

# Leaky: a random split scatters test rows between their own train neighbors.
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X, target, test_size=0.2, random_state=RANDOM_SEED
)
random_split_model = LinearRegression().fit(X_train_r, y_train_r)

# Honest: the test block is entirely after the training block.
cut = int(len(X) * 0.8)
X_train_t, X_test_t = X.iloc[:cut], X.iloc[cut:]
y_train_t, y_test_t = target.iloc[:cut], target.iloc[cut:]
temporal_split_model = LinearRegression().fit(X_train_t, y_train_t)

## 4. Evaluate

*Score against the baseline, not in isolation — a metric only means
something next to what it's beating.*

Here the comparison that matters is split-vs-split, not model-vs-baseline:
the same model, same features, same data — only how it was split changes.

In [ ]:
random_split_r2 = r2_score(y_test_r, random_split_model.predict(X_test_r))
temporal_split_r2 = r2_score(y_test_t, temporal_split_model.predict(X_test_t))

print(f"Random split R^2:   {random_split_r2:.3f}  (looks great — and lies)")
print(f"Temporal split R^2: {temporal_split_r2:.3f}  (the honest number)")

## 5. Interpret

*What is the model actually keying off of? Business-readable, not just
statistically significant.*

The random split's test rows sit *between* their own neighbors in the
training set — with a lag feature, that's almost the answer leaking in
directly, because tomorrow looks a lot like today. The temporal split's test
block has no such neighbors to lean on: it has to extrapolate forward, which
is what deploying the model into the future actually requires.

In [ ]:
fig, ax = plt.subplots()
ax.plot(y_test_t.index, y_test_t.values, label="actual")
ax.plot(y_test_t.index, temporal_split_model.predict(X_test_t), label="predicted (temporal split)")
ax.set_title("Temporal-split test period: prediction vs actual")
ax.legend()
plt.tight_layout()

## 6. Recommend

**The rule:** reach for `gbus564_coursework.utils.sandbox` whenever a demo needs to isolate
*exactly one* failure mode. Dial one knob, leave the rest at their defaults
— mixing noise *and* drift *and* imbalance in the same run muddies which one
caused the failure, defeating the point.

| Function | Knob | Reappears in |
|---|---|---|
| `make_regression_sandbox` | `noise` | Week 2 Lesson 9 — Overfitting in the Sandbox |
| `make_regression_sandbox` | `confound` | Week 3 Lesson 6 — The Multicollinearity Problem |
| `make_regression_sandbox` | small `n_samples`, many `n_features` | Week 6 Lesson 8 — Overfit an NN, then Regularize |
| `make_classification_sandbox` | `noise`, `imbalance` | Week 2 Lessons 5 & 7 — Threshold/Stump; Baselines |
| `make_clusters_sandbox` | `shape` | Week 4 Lesson 12 — Clustering Assumptions |
| `make_timeseries_sandbox` | `drift` | Week 7 Lesson 3 — How Random Splitting Leaks the Future |

**Caveats / next steps:** none of the numbers above are a business result —
there's no stakeholder, no dataset with a real name. The payoff is every
later concept demo skipping this setup and going straight to the failure
mode itself.